# Travelling Salesman Problem via QAOA and PCE

This notebook runs the same TSP three ways: direct constraint-enhanced QAOA, partitioned QAOA, and PCE.


## Problem size

`TSPProblem` fixes the start city, so the register holds **(n−1)² qubits**. Partitioning clusters the cities and runs each cluster's sub-tour as its own QAOA.

| Cities | Qubits | Brute-force permutations |
|--------|--------|-------------------------|
| 3 | 4 | 2 |
| 4 | 9 | 6 |
| 5 | 16 | 24 |
| 9 | 64 | 40,320 |

In [ ]:
import time

from travelling_salesman import (
    compute_distance_matrix,
    extract_best_tour,
    generate_cities,
    plot_cities,
    plot_comparison,
    print_comparison,
    solve_brute_force,
    solve_partitioned_tsp,
    solve_with_pce,
    solve_with_qaoa,
)

from divi.backends import MaestroSimulator

SHOTS = 20_000
SEED = 42

---

## Local example

### Part A — Direct QAOA (4 cities, 9 qubits)

In [ ]:
local_backend = MaestroSimulator(shots=SHOTS)
print("Using local MaestroSimulator")

N_CITIES_SMALL = 4
cities_small = generate_cities(N_CITIES_SMALL, seed=SEED)
dist_small = compute_distance_matrix(cities_small)
print(
    f"\nGenerated {N_CITIES_SMALL} cities ({(N_CITIES_SMALL - 1)**2} qubits)"
)

plot_cities(cities_small, title=f"TSP — {N_CITIES_SMALL} Cities (Direct QAOA)")

In [ ]:
classical_tour_small, classical_dist_small = solve_brute_force(dist_small)
print(
    f"Classical optimum: {classical_tour_small}  distance = {classical_dist_small:.4f}"
)

print("\nRunning direct QAOA locally...")
t0 = time.time()
qaoa = solve_with_qaoa(
    dist_small, n_layers=3, max_iterations=20, shots=SHOTS, backend=local_backend
)
q_tour_small, q_dist_small = extract_best_tour(qaoa)
local_time_a = time.time() - t0

print(f"\nQAOA tour: {q_tour_small}  distance = {q_dist_small:.4f}")
print_comparison(
    classical_tour_small, classical_dist_small, q_tour_small, q_dist_small
)

### Part C — PCE (Pauli Correlation Encoding, qubit compression)

In [ ]:
print(f"Same {N_CITIES_SMALL}-city instance (with qubit compression)")

print("\nRunning PCE-VQE locally...")
t0 = time.time()
pce_tour, pce_dist, pce_qubits = solve_with_pce(
    dist_small,
    n_layers=3,
    max_iterations=20,
    shots=SHOTS,
    backend=local_backend,
)
local_time_c = time.time() - t0

print(f"\nPCE tour: {pce_tour}  distance = {pce_dist:.4f}")
print(f"   Qubits used: {pce_qubits}  (vs {(N_CITIES_SMALL - 1)**2} for direct QAOA)")
print(f"\nDirect QAOA: {local_time_a:.1f}s, PCE: {local_time_c:.1f}s")

---

## Partitioned QAOA (9 cities, 64 qubits)

64 qubits unpartitioned. `TSPProblem(..., max_cities_per_partition=4)` clusters the cities, Divi's `PartitioningProgramEnsemble` runs each cluster's sub-tour, and beam-search aggregation patches the sub-tours into one tour.

In [ ]:
N_CITIES_LARGE = 9
backend = MaestroSimulator(shots=SHOTS)

cities_large = generate_cities(N_CITIES_LARGE, seed=SEED)
dist_large = compute_distance_matrix(cities_large)
print(f"Generated {N_CITIES_LARGE} cities ({(N_CITIES_LARGE - 1)**2} qubits unpartitioned)")

plot_cities(cities_large, title=f"TSP — {N_CITIES_LARGE} Cities (Partitioned QAOA)")

classical_tour_large, classical_dist_large = solve_brute_force(dist_large)
print(
    f"Classical optimum: {classical_tour_large}  distance = {classical_dist_large:.4f}"
)

print(f"\nSolving {N_CITIES_LARGE}-city partitions...")
t0 = time.time()
q_tour_large, q_dist_large = solve_partitioned_tsp(
    dist_large,
    max_cities_per_partition=4,
    n_layers=2,
    max_iterations=10,
    shots=SHOTS,
    backend=backend,
)
partitioned_time = time.time() - t0

print(f"\nPartitioned QAOA tour: {q_tour_large}  distance = {q_dist_large:.4f}")
print_comparison(classical_tour_large, classical_dist_large, q_tour_large, q_dist_large)

plot_comparison(
    cities_large,
    classical_tour_large,
    classical_dist_large,
    q_tour_large,
    q_dist_large,
    save_path="tsp_partitioned_comparison.png",
)

print(
    f"\nDirect QAOA and PCE: {local_time_a + local_time_c:.1f}s for {N_CITIES_SMALL} cities"
)
print(f"Partitioned run: {partitioned_time:.1f}s for {N_CITIES_LARGE} cities")

---

## Further exploration

Try changing the number of cities, partition size, optimiser settings, and shot count. Compare the repaired quantum tours with the classical reference.